# RLHF Pipeline: KL Penalty and Reward Hacking

*Reading list: Ziegler et al. 2019, Ouyang et al. 2022 (InstructGPT), Gao et al. 2022.*

The full RLHF objective, the KL penalty as a distribution-space trust region, and Goodhart dynamics at scale.

In [ ]:
import sys, os
from pathlib import Path
_p = Path.cwd()
while _p != _p.parent and not (_p / 'utils' / 'reward_model.py').exists():
    _p = _p.parent
REPO = str(_p)
sys.path.insert(0, REPO)

from utils.reward_model import BradleyTerry, make_synthetic_preferences, train_rm
from utils.env import ENV
import torch, torch.nn.functional as F, numpy as np, matplotlib.pyplot as plt
ENV.summary()

## 1. The RLHF Objective

The policy $\pi_\theta$ is trained to maximise:

$$r_{\text{total}}(x,y) = r_\phi(x,y) - \beta\, D_{\text{KL}}(\pi_\theta(y|x) \| \pi_{\text{ref}}(y|x))$$

The KL term keeps $\pi_\theta$ close to $\pi_{\text{ref}}$ (the SFT model) in **distribution space**. $\beta$ controls the radius of the allowed region.

**Closed-form optimal policy** (Ziegler et al. 2019, Appendix):

$$\pi^*(y|x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y|x) \exp\!\left(\frac{r_\phi(x,y)}{\beta}\right)$$

where $Z(x)$ is the partition function. This is a Boltzmann distribution over $\pi_{\text{ref}}$ with inverse temperature $1/\beta$.

In [ ]:
# Demonstrate: KL ball geometry
# Vary beta; plot the (KL_budget, expected_reward) frontier
# Toy: 10-action discrete 'policy'; reward = fixed vector
torch.manual_seed(0)
n_actions = 10
r_true = torch.randn(n_actions)           # true reward per action
pi_ref = torch.softmax(torch.zeros(n_actions), dim=0)  # uniform reference

betas = np.logspace(-1, 1.5, 60)
kl_vals, reward_vals = [], []

for beta in betas:
    # Optimal policy in closed form
    log_pi_star = torch.log(pi_ref) + r_true / beta
    pi_star = torch.softmax(log_pi_star, dim=0)
    kl = (pi_star * (pi_star.log() - pi_ref.log())).sum().item()
    reward = (pi_star * r_true).sum().item()
    kl_vals.append(kl); reward_vals.append(reward)

plt.figure(figsize=(6,3.5))
plt.plot(kl_vals, reward_vals, 'o-', ms=3)
plt.xlabel('KL($\\pi^*$ || $\\pi_{ref}$)')
plt.ylabel('Expected reward')
plt.title('KL-reward frontier (β sweeps right→left)')
for b, kl, rew in zip(betas[::15], kl_vals[::15], reward_vals[::15]):
    plt.annotate(f'β={b:.1f}', (kl, rew), textcoords='offset points',
                 xytext=(5,3), fontsize=7)
plt.tight_layout(); plt.show()

## 2. The KL Budget Tradeoff

As $\beta \to 0$: policy optimises reward freely → reward hacking.  
As $\beta \to \infty$: policy stays at $\pi_{\text{ref}}$ → no improvement.  

The optimal $\beta$ depends on reward model quality — a better RM tolerates lower $\beta$ before Goodhart kicks in (Gao et al. 2022).

In [ ]:
# Simulate: proxy RM (noisy), true reward, sweep beta
torch.manual_seed(1)
n = 20
r_true_vec  = torch.randn(n)
noise_level = 0.5
r_proxy_vec = r_true_vec + noise_level * torch.randn(n)
pi_ref_vec  = torch.softmax(torch.zeros(n), dim=0)

betas = np.logspace(-1, 2, 80)
true_rewards, proxy_rewards, kls = [], [], []

for beta in betas:
    log_pi = torch.log(pi_ref_vec) + r_proxy_vec / beta
    pi = torch.softmax(log_pi, dim=0)
    kl = (pi*(pi.log()-pi_ref_vec.log())).sum().item()
    true_rewards.append((pi*r_true_vec).sum().item())
    proxy_rewards.append((pi*r_proxy_vec).sum().item())
    kls.append(kl)

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,3.5))
ax1.plot(kls, proxy_rewards, label='proxy RM', color='blue')
ax1.plot(kls, true_rewards,  label='true reward', color='red')
ax1.set_xlabel('KL from reference'); ax1.set_ylabel('Expected reward')
ax1.set_title('Goodhart curve: proxy diverges from true reward')
ax1.legend()
ax2.semilogx(betas, true_rewards, color='red')
ax2.set_xlabel('β'); ax2.set_ylabel('True reward')
ax2.set_title('True reward vs β — peak at intermediate β')
plt.tight_layout(); plt.show()

## 3. The Partition Function and Free Energy

The optimal RLHF policy has the form of a Boltzmann distribution:

$$\pi^*(y|x) \propto \pi_{\text{ref}}(y|x)\, e^{r(y)/\beta}$$

This is exactly the Gibbs distribution from statistical mechanics with:
- **Energy**: $E(y) = -r(y)$ (reward = negative energy)
- **Temperature**: $\beta$ (higher β = higher temperature = more entropy)
- **Partition function**: $Z(x) = \sum_y \pi_{\text{ref}}(y|x) e^{r(y)/\beta}$

The free energy $F = -\beta \log Z$ is the RLHF objective value at optimum:
$J^* = \beta \log Z = $ (expected reward under $\pi^*$) $- \beta \cdot D_{KL}(\pi^* \| \pi_{ref})$.

In [ ]:
# Verify: free energy relation
beta = 1.0
log_Z = torch.logsumexp(torch.log(pi_ref_vec) + r_true_vec/beta, dim=0)
log_pi_star = torch.log(pi_ref_vec) + r_true_vec/beta - log_Z
pi_star = log_pi_star.exp()

kl    = (pi_star*(log_pi_star - pi_ref_vec.log())).sum()
er    = (pi_star * r_true_vec).sum()
J_opt = er - beta * kl    # RLHF objective at optimum
F     = beta * log_Z      # free energy

print(f'E[r] under pi*:          {er.item():.4f}')
print(f'KL(pi* || pi_ref):       {kl.item():.4f}')
print(f'J* = E[r] - beta*KL:     {J_opt.item():.4f}')
print(f'beta * log Z (free energy): {F.item():.4f}')
print(f'Equal: {torch.isclose(J_opt, F)}')